# Building a RAG System with LangChain + FAISS, Gemini Embeddings, and Groq Generation

## A source-faithful, interview-ready learning notebook

This notebook follows the instructor's original FAISS lesson and both transcript files, while improving the explanations and correcting implementation problems. It teaches the full path from source documents to embeddings, FAISS storage, retrieval, and three RAG patterns: **simple LCEL RAG**, **streaming RAG**, and **conversational RAG**.

### Provider separation is intentional

- **Gemini API** is used only to create text embeddings for semantic retrieval.
- **Groq API** is used only to generate the natural-language RAG answer.
- **FAISS** is the local vector-index/search library; it is not a language model and does not call an API itself.
- **LangChain** connects documents, embeddings, retrieval, prompts, and generation.

No OpenAI API key or OpenAI API client is used. The Groq model selected below has `openai/` in its provider-side model ID, but requests still go through Groq using `GROQ_API_KEY`; this does not mean the OpenAI API is called.

> **How to study:** read each Markdown cell before running the code cell immediately below it. After each section, explain what it does, why it exists, what enters it, and what comes out. The live API cells can consume quota.

## Learning outcomes and lesson map

By the end, you should be able to explain and implement:

1. RAG and its retrieval → context → generation flow.
2. LangChain `Document`, metadata, chunking, `chunk_size`, and `chunk_overlap`.
3. The difference between `embed_query()` and `embed_documents()`.
4. Gemini Embedding 2 task prefixes and why query/document formatting must be consistent.
5. Cosine similarity by formula, including the zero-vector edge case.
6. What FAISS does, how a vector index is created, and what this particular notebook's default index does.
7. Save/load persistence, similarity search, distance scores, metadata filters, and retrievers.
8. LCEL composition with `RunnablePassthrough`, document formatting, prompts, Groq, and `StrOutputParser`.
9. Token streaming and passing chat history into a conversational chain.
10. The safety limits and operational trade-offs of this tutorial's implementation.

### The pipeline we build

`Source text → Document objects + metadata → chunks → Gemini document vectors → FAISS index → query vector → top-k chunks → prompt + context → Groq answer`

For a follow-up question, we also pass `chat_history` into the conversational prompt. This notebook preserves the instructor's simple conversational chain; it does **not** rewrite the follow-up into a standalone retrieval query.

## Source coverage: what was retained and what was improved

This notebook is based on the uploaded `2-faiss.ipynb`, `Pasted text(20261026-085833).txt`, and `Pasted text (2)(20261026-085932).txt`.

- Retained: FAISS overview; CPU/GPU distinction; dense vectors and nearest-neighbor search; sample `Document` records; text splitting; one-query and multi-document embeddings; cosine comparison examples; FAISS creation; save/load; top-k search; score display; metadata filter; retriever; simple RAG; streaming; conversational history; test questions; follow-up question.
- Replaced: OpenAI embeddings and OpenAI environment variable setup with Gemini Embedding 2; Groq generation remains a separate provider section, with an explicit current model setting.
- Corrected: recursive separators, `search_kwargs` spelling, misleading score wording, model-name/provider confusion, and untrusted-pickle warning.
- Added: explicit explanations before code, optional Text/PDF loader demonstration, guarded zero-vector handling, embedding adapter/cache, source audit, troubleshooting, and interview-ready definitions.
- Not implemented here: FAISS GPU indexes, specialized approximate indexes (IVF/HNSW/PQ), large-scale benchmarking, persistent conversation memory, and a history-aware query-rewrite retriever. They are explained as boundaries or next steps, not claimed as built.

The instructor's sample code is preserved in purpose and teaching progression, but provider-specific OpenAI calls and erroneous/redundant statements are intentionally not kept as executable code.

## 1. Prepare the Python environment

Run this command in **Windows CMD or your project terminal**, not inside a Python cell. It installs FAISS CPU, the Gemini SDK, LangChain interfaces, the Groq integration, text splitting, PDF support, and environment-variable loading.

```cmd
uv add google-genai langchain-core langchain-community langchain-text-splitters langchain-groq python-dotenv numpy faiss-cpu pypdf
```

`faiss-cpu` is sufficient for the CPU example in this notebook. FAISS also has GPU implementations, but this notebook does not configure a GPU index. Current package wheels depend on your Python/platform combination; if installation fails, check that your Python version has a compatible wheel before changing the rest of the code.

Official install/project references:
- FAISS: https://github.com/facebookresearch/faiss
- LangChain FAISS integration: https://docs.langchain.com/oss/python/integrations/vectorstores/faiss
- `faiss-cpu` on PyPI: https://pypi.org/project/faiss-cpu/

## 2. Configure credentials safely

Create a `.env` file in your project root with these two variables:

```env
GEMINI_API_KEY=your_gemini_api_key_here
GROQ_API_KEY=your_groq_api_key_here
```

Replace the placeholders in your local file. Never paste real credentials into notebook source, print them, or commit `.env` to Git. Add `.env`, `faiss_index_gemini_groq/`, and any local cache or data directories to `.gitignore`.

The next cell loads the file only. The **Gemini section validates only the Gemini key**, and the **Groq section validates only the Groq key**, so provider setup and errors stay separate.

In [1]:
from pathlib import Path
import os
from dotenv import find_dotenv, load_dotenv

# Search from the notebook's current working directory and parent directories.
env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
else:
    # Also supports environment variables already set in the shell/kernel.
    load_dotenv()

print("Environment file:",
      env_path if env_path else "not found; shell variables may be in use")

Environment file: e:\RAG\.env


## 3. Import shared LangChain, FAISS, and data utilities

These are shared building blocks, not provider calls:

- `Document` keeps text and metadata together.
- `RecursiveCharacterTextSplitter` creates retrieval-sized passages.
- `FAISS` is LangChain's wrapper around the FAISS index plus a document store and ID mapping.
- `ChatPromptTemplate`, message types, runnable primitives, and `StrOutputParser` are used later in the Groq generation sections.
- `TextLoader` and `PyPDFLoader` are optional ingestion demonstrations. The main index remains based on the instructor's four in-memory sample documents, so the same corpus is used throughout the lesson.

We deliberately do not suppress all Python warnings. Broad warning suppression can hide compatibility and deprecation signals that are important when libraries change.

In [2]:
import numpy as np
import faiss

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader

print("Shared imports completed.")

Shared imports completed.


C:\Users\sniti\AppData\Local\Temp\ipykernel_10460\1980569006.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 4. Check the FAISS runtime

This is a lightweight diagnostic. It confirms that the Python package imported and prints the NumPy/FAISS versions when available. It does not benchmark speed or prove that a production-scale index will fit in memory.

FAISS supports multiple index structures and metrics. **The LangChain `FAISS.from_documents()` path used later defaults to an exact flat L2 index in this setup.** That is a good baseline for a small teaching corpus; it is not an implementation of every FAISS index algorithm.

In [3]:
print("FAISS module imported:", faiss.__name__)
print("FAISS version:", getattr(
    faiss, "__version__", "version attribute not exposed"))
print("NumPy version:", np.__version__)
print("CPU index demo: yes; GPU index configured: no")

FAISS module imported: faiss
FAISS version: 1.15.1
NumPy version: 2.5.3
CPU index demo: yes; GPU index configured: no


## 5. Understand FAISS before coding

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search and clustering over dense vectors. It stores/searches vectors using an index structure. Similarity is commonly calculated through L2 (Euclidean) distance or inner product; cosine similarity can be computed as inner product after normalizing vectors.

### Why use it?

- Fast nearest-neighbor search over numeric vectors.
- Multiple choices that trade retrieval speed, memory, build time, and exactness.
- CPU support and separate GPU-capable implementations.
- Can support large vector collections depending on index type, memory, hardware, and configuration.

### What this notebook actually uses

The example uses `faiss-cpu` and LangChain's default flat L2 index. This is an exact-search baseline. It does not demonstrate GPU execution, IVF training, HNSW, product quantization, sharding, multi-tenancy, or a managed database server. FAISS is a library/index, not a full managed vector database with built-in service-level operations.

Official description: https://github.com/facebookresearch/faiss

## 6. Create the instructor's sample documents

A LangChain `Document` has two key fields:

- `page_content`: the actual text that will be split, embedded, and eventually shown to the answer model.
- `metadata`: structured fields such as `source`, `page`, and `topic`, useful for attribution, filtering, and debugging.

We retain the instructor's four sample topics: AI, machine learning, deep learning, and NLP. This deliberately small corpus makes returned results easy to inspect by eye.

In [4]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """.strip(),
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"},
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """.strip(),
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"},
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """.strip(),
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"},
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """.strip(),
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"},
    ),
]

print(f"Created {len(sample_documents)} sample Document objects.")

Created 4 sample Document objects.


## 7. Inspect the original `Document` objects

Inspect at least one object before splitting. This helps distinguish content from metadata and makes later source attribution less mysterious. Printing every full object is fine for a four-document demo, but production systems should avoid dumping large or sensitive document bodies into logs.

In [5]:
for i, doc in enumerate(sample_documents, start=1):
    print(f"\n--- Document {i} ---")
    print("Source:", doc.metadata.get("source"))
    print("Topic:", doc.metadata.get("topic"))
    print("Content:", doc.page_content)


--- Document 1 ---
Source: AI Introduction
Topic: AI
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.

--- Document 2 ---
Source: ML Basics
Topic: ML
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.

--- Document 3 ---
Source: Deep Learning
Topic: DL
Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.

--- Document 4 ---
Source: NLP Overview
Topic: NLP
Content: Natural Language Processi

## 8. Optional: understand text and PDF loaders

The instructor imports `TextLoader` and `PyPDFLoader` as possible ingestion tools. The core FAISS pipeline below uses the in-memory `Document` objects above, but this optional section demonstrates how files can become `Document` objects too.

- `TextLoader` reads a plain-text file and returns documents.
- `PyPDFLoader` reads PDF pages, typically keeping page metadata.
- PDF loading depends on the installed `pypdf` package.

This demonstration does **not** append these sample files to the main corpus, because that would duplicate content in the vector index.

In [6]:
DATA_DIR = Path("data/faiss_gemini_groq")
DATA_DIR.mkdir(parents=True, exist_ok=True)

text_demo_path = DATA_DIR / "loader_demo.txt"
text_demo_path.write_text(sample_documents[0].page_content, encoding="utf-8")
loaded_text_documents = TextLoader(
    str(text_demo_path), encoding="utf-8").load()
print("TextLoader returned:", len(loaded_text_documents), "document(s)")
print("Loaded source metadata:", loaded_text_documents[0].metadata)

# Optional PDF demonstration: put your own PDF at this path to run the loader.
PDF_PATH = DATA_DIR / "your_document.pdf"
if PDF_PATH.exists():
    loaded_pdf_documents = PyPDFLoader(str(PDF_PATH)).load()
    print("PyPDFLoader returned page documents:", len(loaded_pdf_documents))
    print("First page metadata:",
          loaded_pdf_documents[0].metadata if loaded_pdf_documents else {})
else:
    print(f"PDF loader demo skipped: add a PDF at {PDF_PATH} to try it.")

TextLoader returned: 1 document(s)
Loaded source metadata: {'source': 'data\\faiss_gemini_groq\\loader_demo.txt'}
PDF loader demo skipped: add a PDF at data\faiss_gemini_groq\your_document.pdf to try it.


## 9. Split documents into chunks

A **chunk** is a smaller passage derived from a source document. Vector retrieval typically retrieves passages rather than entire large files because focused passages make the context easier to rank and fit into the prompt.

- `chunk_size=500`: target maximum passage size in characters because `length_function=len` is used here.
- `chunk_overlap=50`: repeats a small region between adjacent chunks to reduce boundary loss.
- `add_start_index=True`: records the chunk's start character position in metadata.
- Recursive separators first prefer paragraph and line boundaries, then spaces, and finally character boundaries. The instructor's `separators=[" "]` was overly restrictive; we keep the teaching intent but restore meaningful recursive splitting.

The sample passages are short, so the main lesson may produce one chunk per document. Real long files are where chunk sizing and overlap become more visible.

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
    add_start_index=True,
)

chunks = text_splitter.split_documents(sample_documents)
if not chunks:
    raise ValueError("Text splitting produced no chunks.")

print(
    f"Created {len(chunks)} chunks from {len(sample_documents)} source documents.")

Created 4 chunks from 4 source documents.


## 10. Inspect chunk content and metadata

A good ingestion check confirms that chunk text is non-empty and source/topic metadata has survived splitting. Metadata is copied to child chunks, so retrieval results can still be traced to the source document.

In [8]:
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print("Character count:", len(chunk.page_content))
    print("Metadata:", chunk.metadata)
    print("Content:", chunk.page_content[:350])


--- Chunk 1 ---
Character count: 223
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI', 'start_index': 0}
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.

--- Chunk 2 ---
Character count: 243
Metadata: {'source': 'ML Basics', 'page': 1, 'topic': 'ML', 'start_index': 0}
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.

--- Chunk 3 ---
Character count: 264
Metadata: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL', 'start_index': 0}
Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-

## 11. Gemini section: validate the embedding API key

This section uses the **Gemini API only**. Gemini turns text into numeric vectors used for retrieval; it does not generate our final RAG answer in this notebook.

The instructor used OpenAI embeddings in the source notebook. Here, their role is replaced by Gemini Embedding 2 while keeping the same teaching goals: one query embedding, a batch of document embeddings, vector inspection, cosine similarity, and integration with FAISS.

Google recommends these text formats for asymmetric retrieval with Gemini Embedding 2:

- Document: `title: none | text: ...`
- Query: `task: question answering | query: ...`

Gemini Embedding 2 uses task instructions in the text format; do not copy the older `task_type` argument examples for `gemini-embedding-001` into this code.

Documentation: https://ai.google.dev/gemini-api/docs/embeddings

### 11A. Initialize Gemini independently

This cell reads only `GEMINI_API_KEY` and initializes the Gemini SDK client. If this cell fails, fix the Gemini key/network/quota before moving to embeddings. No Groq model is initialized here.

In [9]:
from google import genai
from langchain_core.embeddings import Embeddings

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY is missing. Add it to .env or your environment.")

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_EMBEDDING_MODEL = "gemini-embedding-2"

print("Gemini embedding client initialized.")
print("Gemini embedding model:", GEMINI_EMBEDDING_MODEL)
print("API key value is hidden.")

Gemini embedding client initialized.
Gemini embedding model: gemini-embedding-2
API key value is hidden.


## 12. Generate one embedding directly with Gemini

Before integrating an API into LangChain, test a single request directly. This isolates credential, model-availability, network, and response-shape issues from FAISS or chain issues.

`embed_content` returns an embedding vector for the formatted input. A vector is a numeric representation—not an answer and not a list of keywords.

In [10]:
sample_text = "What is machine learning?"
formatted_query = f"task: question answering | query: {sample_text}"

embedding_response = gemini_client.models.embed_content(
    model=GEMINI_EMBEDDING_MODEL,
    contents=formatted_query,
)

sample_embedding = embedding_response.embeddings[0].values
if not sample_embedding:
    raise ValueError("Gemini returned an empty embedding.")

print("Input:", sample_text)
print("Vector dimension:", len(sample_embedding))
print("First 5 values:", [round(float(x), 6) for x in sample_embedding[:5]])

Input: What is machine learning?
Vector dimension: 3072
First 5 values: [-0.014944, -0.004446, 0.00492, 0.024048, -0.000857]


## 13. Adapt Gemini to LangChain's `Embeddings` interface

LangChain vector stores expect an embedding object with two methods:

- `embed_documents(texts)`: create one vector per source text at indexing time.
- `embed_query(text)`: create one vector for the user's search query at retrieval time.

This adapter intentionally sends one formatted text per Gemini request to make the query/document difference obvious and avoid confusing Gemini Embedding 2's aggregated multimodal-input behavior with a list of independent documents. Its in-memory dictionary avoids repeating identical requests within this Python kernel. The cache is **not persistent** and does not survive a kernel restart.

The same model and formatting must be used when building the index and querying it. Changing model/version/prefix means you should rebuild the index rather than mixing incompatible vectors.

In [11]:
DOCUMENT_PREFIX = "title: none | text: "
QUERY_PREFIX = "task: question answering | query: "


class GeminiEmbeddings(Embeddings):
    """Small LangChain adapter around the Gemini Embedding 2 API."""

    def __init__(self, client, model: str = "gemini-embedding-2"):
        self.client = client
        self.model = model
        self._cache: dict[str, list[float]] = {}
        self._dimension: int | None = None

    def _embed_formatted_text(self, formatted_text: str) -> list[float]:
        if formatted_text in self._cache:
            return list(self._cache[formatted_text])

        response = self.client.models.embed_content(
            model=self.model,
            contents=formatted_text,
        )
        if not response.embeddings or not response.embeddings[0].values:
            raise ValueError("Gemini returned no embedding values.")

        vector = [float(value) for value in response.embeddings[0].values]
        if self._dimension is None:
            self._dimension = len(vector)
        elif len(vector) != self._dimension:
            raise ValueError(
                f"Embedding dimension changed from {self._dimension} to {len(vector)}. "
                "Do not mix incompatible vectors in one FAISS index."
            )

        self._cache[formatted_text] = vector
        return list(vector)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embed each document separately using the document format."""
        return [self._embed_formatted_text(DOCUMENT_PREFIX + text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        """Embed a search query using the query-specific format."""
        return self._embed_formatted_text(QUERY_PREFIX + text)


gemini_embeddings = GeminiEmbeddings(
    gemini_client, model=GEMINI_EMBEDDING_MODEL)
print("GeminiEmbeddings adapter ready for LangChain.")

GeminiEmbeddings adapter ready for LangChain.


## 14. Test the LangChain embedding adapter

This uses the same adapter that FAISS will call. We check that the query vector is non-empty and show its dimension and a short preview. The adapter keeps the output as Python floats, which is the representation expected by LangChain's vector-store interface.

In [12]:
adapter_query = "What is deep learning?"
adapter_vector = gemini_embeddings.embed_query(adapter_query)

assert isinstance(adapter_vector, list) and len(adapter_vector) > 0
assert all(isinstance(value, float) for value in adapter_vector)

print("Query:", adapter_query)
print("Dimensions:", len(adapter_vector))
print("Vector preview:", [round(v, 5) for v in adapter_vector[:5]])

Query: What is deep learning?
Dimensions: 3072
Vector preview: [-0.01693, 0.01485, 0.01038, 0.02769, -0.00739]


## 15. Create embeddings for a list of texts

This retains the instructor's `embed_documents()` demonstration using AI, machine learning, deep learning, and neural networks. The adapter makes one embedding per input text, preserving the one-text-to-one-vector mapping.

For large corpora, individual requests can be slower and may hit rate limits. Production ingestion should investigate the provider's batch options, retry/backoff, rate-limit handling, and persistent caching. Those production features are outside this notebook's small local demo.

In [13]:
texts_for_embedding = [
    "AI",
    "Machine learning",
    "Deep learning",
    "Neural networks",
]

batch_embeddings = gemini_embeddings.embed_documents(texts_for_embedding)
assert len(batch_embeddings) == len(texts_for_embedding)

print("Text count:", len(texts_for_embedding))
print("Vector count:", len(batch_embeddings))
print("Vector dimensions:", sorted({len(vector)
      for vector in batch_embeddings}))

Text count: 4
Vector count: 4
Vector dimensions: [3072]


## 16. Inspect example embeddings

The values of an embedding vector are difficult for a human to interpret directly. The useful operation is comparing vectors or searching an index—not trying to assign a plain-English meaning to one coordinate.

In [14]:
for text, vector in zip(texts_for_embedding, batch_embeddings):
    print(
        f"{text:20s} | dimensions={len(vector)} | first 4={[round(v, 5) for v in vector[:4]]}")

AI                   | dimensions=3072 | first 4=[-0.01627, -0.01087, -0.00602, 0.01664]
Machine learning     | dimensions=3072 | first 4=[-0.03003, -0.02373, 0.00195, 0.02809]
Deep learning        | dimensions=3072 | first 4=[-0.02844, -0.00638, 0.0085, 0.03367]
Neural networks      | dimensions=3072 | first 4=[-0.00568, -0.00227, 0.01096, 0.02443]


## 17. Cosine similarity: definition and formula

**Cosine similarity** compares the angle between two non-zero vectors:

\[
\operatorname{cosine}(a,b)=\frac{a\cdot b}{\|a\|\,\|b\|}
\]

For normalized vectors the score corresponds to their dot product. A larger score generally indicates a smaller angle and stronger directional similarity. The score is **not automatically a probability or a percentage of semantic truth**. A zero vector makes the denominator zero, so the function below explicitly rejects that edge case.

In [15]:
def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """Return cosine similarity for two same-length, non-zero vectors."""
    a = np.asarray(vec_a, dtype=np.float64)
    b = np.asarray(vec_b, dtype=np.float64)

    if a.ndim != 1 or b.ndim != 1 or a.shape != b.shape:
        raise ValueError(
            "Both vectors must be one-dimensional and have the same length.")

    norm_a = float(np.linalg.norm(a))
    norm_b = float(np.linalg.norm(b))
    denominator = norm_a * norm_b
    if denominator == 0.0:
        raise ValueError("Cosine similarity is undefined for a zero vector.")

    return float(np.dot(a, b) / denominator)

## 18. Compare semantic similarity: related and unrelated examples

The instructor compares `AI` with `Artificial Intelligence`, `AI` with `Pizza`, and `Machine Learning` with `ML`. The small in-memory cache in the adapter prevents duplicate requests when the same formatted query occurs more than once.

Actual values depend on the embedding model/version and text formatting. Do not memorize a specific numeric score as a universal fact; interpret the scores comparatively within the same embedding space.

In [16]:
def compare_embeddings(text1: str, text2: str) -> float:
    """Compare two text inputs by embedding both as query-style text."""
    vector1 = gemini_embeddings.embed_query(text1)
    vector2 = gemini_embeddings.embed_query(text2)
    return cosine_similarity(vector1, vector2)


pairs = [
    ("AI", "Artificial Intelligence"),
    ("AI", "Pizza"),
    ("Machine Learning", "ML"),
]

for text1, text2 in pairs:
    score = compare_embeddings(text1, text2)
    print(f"{text1!r} vs {text2!r}: cosine similarity = {score:.4f}")

'AI' vs 'Artificial Intelligence': cosine similarity = 0.9247
'AI' vs 'Pizza': cosine similarity = 0.7793
'Machine Learning' vs 'ML': cosine similarity = 0.8507


## 19. Create the FAISS vector store

Now the documents are ready, split, and paired with a Gemini embedding interface. `FAISS.from_documents()` embeds each chunk and builds the search index; LangChain also maintains a document store and a mapping from FAISS index positions to document IDs.

The core requirement is **dimension consistency**: every document vector and query vector must have the same dimension and come from a compatible embedding space. In this lesson, LangChain's standard FAISS wrapper builds the default flat L2 index. We are not manually configuring a GPU or approximate index.

In [17]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=gemini_embeddings,
)

print("FAISS vector store created.")
print("Vectors in index:", vectorstore.index.ntotal)
print("Chunks provided:", len(chunks))
assert vectorstore.index.ntotal == len(chunks)

FAISS vector store created.
Vectors in index: 4
Chunks provided: 4


## 20. Inspect the built index

`vectorstore.index.ntotal` reports the number of vectors in the FAISS index. It is a simple integrity check that all expected chunks were indexed. It does not tell us whether the returned passages are relevant; we test that separately with queries.

In [18]:
print("Index type:", type(vectorstore.index).__name__)
print("Vector count:", vectorstore.index.ntotal)
print("Embedding dimension:", vectorstore.index.d)

Index type: IndexFlatL2
Vector count: 4
Embedding dimension: 3072


## 21. Save the FAISS index for reuse

`save_local()` writes a local representation so the index does not need to be rebuilt every time. In the LangChain FAISS implementation, the directory commonly contains the FAISS index file and a pickle file containing the document store/mapping. This is not simply “the index stored in a pickle” — the FAISS vector index and its supporting Python data are separate artifacts.

Use a dedicated path for this notebook so it does not overwrite another project's index.

In [19]:
FAISS_INDEX_DIR = Path("faiss_index_gemini_groq")
vectorstore.save_local(str(FAISS_INDEX_DIR))

print("FAISS index saved to:", FAISS_INDEX_DIR.resolve())
print("Saved files:", sorted(path.name for path in FAISS_INDEX_DIR.iterdir()))

FAISS index saved to: E:\RAG\Vector_stores_&_Vector_databases\faiss_index_gemini_groq
Saved files: ['index.faiss', 'index.pkl']


## 22. Load the saved index — important pickle security warning

The instructor uses `allow_dangerous_deserialization=True`, which LangChain requires for loading the saved pickle in this path. **Only enable it for an index directory you created yourself or otherwise fully trust.** Loading a malicious pickle can execute arbitrary code. Never download a random FAISS folder and load its pickle just because it is called an index.

The embedding adapter passed at load time must be compatible with the model and query formatting used to build the vectors. A different embedding space requires a new index.

In [20]:
loaded_vectorstore = FAISS.load_local(
    str(FAISS_INDEX_DIR),
    gemini_embeddings,
    # Safe only for a trusted, self-created artifact.
    allow_dangerous_deserialization=True,
)

print("Loaded FAISS index successfully.")
print("Loaded vector count:", loaded_vectorstore.index.ntotal)

Loaded FAISS index successfully.
Loaded vector count: 4


## 23. Verify the loaded index

Always verify persistence rather than trusting that `load_local()` returned an object. Compare the vector count and then run a search using the loaded vector store.

In [21]:
assert loaded_vectorstore.index.ntotal == vectorstore.index.ntotal
print("Original count:", vectorstore.index.ntotal)
print("Loaded count:  ", loaded_vectorstore.index.ntotal)
print("Persistence count check passed.")

Original count: 4
Loaded count:   4
Persistence count check passed.


## 24. Run basic top-k similarity search

A **similarity search** embeds the text query and retrieves the nearest stored vectors. `k=3` means “return up to three matching documents,” not “return three documents whose relevance exceeds a particular threshold.”

This is the instructor's original example: query “What is deep learning?” and inspect the top three returned chunks.

In [22]:
query = "What is deep learning?"
results = loaded_vectorstore.similarity_search(query, k=3)

print("Query:", query)
print("Results returned:", len(results))
for rank, doc in enumerate(results, start=1):
    print(f"\n--- Result {rank} ---")
    print("Source:", doc.metadata.get("source", "Unknown"))
    print("Topic:", doc.metadata.get("topic", "Unknown"))
    print("Content:", doc.page_content[:250])

Query: What is deep learning?
Results returned: 3

--- Result 1 ---
Source: Deep Learning
Topic: DL
Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speec

--- Result 2 ---
Source: ML Basics
Topic: ML
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.

--- Result 3 ---
Source: NLP Overview
Topic: NLP
Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sent


## 25. Similarity search with scores

`similarity_search_with_score()` returns `(Document, score)` pairs. The default FAISS index used here is L2-based, and these scores are **distances** (for a flat L2 index, FAISS reports squared L2 distance). Lower distance means closer vectors under this index configuration.

Do not call these values “percentage similarity.” Score direction and meaning depend on the index/metric. If you need cosine similarity, normalize vectors and configure the relevant index/search semantics deliberately; don't assume that a default L2 distance is cosine similarity.

In [23]:
results_with_scores = loaded_vectorstore.similarity_search_with_score(
    query, k=3)

for rank, (doc, distance) in enumerate(results_with_scores, start=1):
    print(f"\n--- Result {rank} | L2 distance: {distance:.4f} ---")
    print("Source:", doc.metadata.get("source", "Unknown"))
    print("Topic:", doc.metadata.get("topic", "Unknown"))
    print("Content preview:", doc.page_content[:180])


--- Result 1 | L2 distance: 0.3924 ---
Source: Deep Learning
Topic: DL
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
  

--- Result 2 | L2 distance: 0.6585 ---
Source: ML Basics
Topic: ML
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common type

--- Result 3 | L2 distance: 0.7248 ---
Source: NLP Overview
Topic: NLP
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep lea


## 26. Filter search by metadata

Metadata filtering restricts retrieval to records whose structured fields satisfy a condition. It is useful when you want to search only one topic, source, tenant, product, date range, or access scope.

The instructor's example is `{"topic": "ML"}`. The filter matches the exact `topic` metadata value stored on the source document. FAISS's LangChain integration may first retrieve a candidate set and then apply the filter, so `fetch_k` can affect how many filtered results remain.

In [24]:
filter_dict = {"topic": "ML"}
filtered_results = loaded_vectorstore.similarity_search(
    query=query,
    k=3,
    filter=filter_dict,
    fetch_k=20,
)

print("Filter:", filter_dict)
print("Filtered results:", len(filtered_results))
for rank, doc in enumerate(filtered_results, start=1):
    print(f"\n--- Filtered result {rank} ---")
    print("Topic:", doc.metadata.get("topic"))
    print("Source:", doc.metadata.get("source"))
    print("Content:", doc.page_content[:200])

Filter: {'topic': 'ML'}
Filtered results: 1

--- Filtered result 1 ---
Topic: ML
Source: ML Basics
Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised


## 27. Convert the vector store into a retriever

A **vector store** exposes storage and search functions; a **retriever** is a standard component that accepts a query and returns `Document` objects. This abstraction lets the same RAG chain work with different retrieval backends.

`search_type="similarity"` means nearest-vector similarity search. `search_kwargs={"k": 3}` asks for up to three matching chunks.

In [25]:
retriever = loaded_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

retrieved_docs = retriever.invoke("How does NLP work?")
print("Retriever returned:", len(retrieved_docs), "document(s)")
for rank, doc in enumerate(retrieved_docs, start=1):
    print(f"{rank}. {doc.metadata.get('source', 'Unknown')} | topic={doc.metadata.get('topic', 'Unknown')}")

Retriever returned: 3 document(s)
1. NLP Overview | topic=NLP
2. Deep Learning | topic=DL
3. ML Basics | topic=ML


## 28. Groq section: initialize the answer-generation model separately

This is the **Groq API only** section. It does not create embeddings or call Gemini. Groq provides the chat completion used to write the final answer after FAISS retrieval.

### Model ID note

This notebook uses `openai/gpt-oss-120b` as the **model ID served by Groq** through `ChatGroq`. The `openai/` prefix identifies the model family/provider namespace; our request is still sent to Groq with `GROQ_API_KEY`, not to OpenAI's API. Model availability and account limits can change, so verify the current Groq model list if the configured ID is unavailable.

- Groq supported models: https://console.groq.com/docs/models
- Groq model details: https://console.groq.com/docs/model/openai/gpt-oss-120b

No OpenAI package or `OPENAI_API_KEY` is used.

### 28A. Validate the Groq key and initialize Groq

Keep this cell independent from the Gemini initialization cell so API failures are easier to diagnose. If it fails, check `GROQ_API_KEY`, model access, quota/rate limits, and network access.

In [26]:
from langchain_groq import ChatGroq

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise EnvironmentError(
        "GROQ_API_KEY is missing. Add it to .env or your environment.")

GROQ_MODEL = "openai/gpt-oss-120b"
groq_llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0.2,
)

print("Groq chat model initialized.")
print("Groq model ID:", GROQ_MODEL)
print("Provider API key value is hidden.")

Groq chat model initialized.
Groq model ID: openai/gpt-oss-120b
Provider API key value is hidden.


## 29. Test Groq generation without RAG

This direct call verifies the generation model alone. It uses no FAISS retrieval and no Gemini embedding call. Separating this check helps identify whether an error comes from the provider/model or from the RAG chain.

In [27]:
groq_test_response = groq_llm.invoke(
    "Explain retrieval-augmented generation (RAG) in two simple sentences."
)
print(groq_test_response.content)

Retrieval‑augmented generation (RAG) combines a language model with a searchable knowledge base, first pulling relevant documents or passages and then using them as context to produce the final answer. This lets the system generate up‑to‑date, factual responses even when the model’s own training data is limited or outdated.


## 30. Create the basic RAG prompt

A prompt template has two runtime variables in this implementation:

- `{context}`: the retrieved chunks formatted into readable text.
- `{question}`: the user's current question.

The prompt instructs the model to answer using the context and to admit when the context is insufficient. This reduces unsupported answers, but it does not guarantee truth; retrieval quality and evaluation still matter.

In [28]:
simple_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context.
If the context does not contain enough information, clearly say that the provided documents are insufficient.
Do not invent facts.

Context:
{context}

Question: {question}

Answer:"""
)

## 31. Format retrieved `Document` objects for the prompt

The retriever returns a list of `Document` objects. The language model prompt expects a string for `{context}`, so `format_docs()` combines each passage with its source/topic label.

Keeping source labels in the context makes debugging and source inspection easier. The labels in this basic prompt do not constitute a full citation-verification system.

In [29]:
def format_docs(docs: list[Document]) -> str:
    """Combine retrieved chunks into a readable prompt context string."""
    formatted = []
    for index, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "Unknown source")
        topic = doc.metadata.get("topic", "Unknown topic")
        formatted.append(
            f"Document {index} (Source: {source}; Topic: {topic}):\n{doc.page_content}"
        )
    return "\n\n".join(formatted)

## 32. Build the simple LCEL RAG chain

**LCEL** (LangChain Expression Language) composes components using `|`.

The mapping at the start creates two prompt inputs in parallel:

- `"context": retriever | format_docs` sends the input question to the retriever, then formats the returned documents.
- `"question": RunnablePassthrough()` passes the original question through unchanged.

The resulting dictionary fills the prompt variables; the Groq chat model generates a response; `StrOutputParser()` converts the model message to plain text. This is the same conceptual flow demonstrated in the instructor's FAISS transcript.

In [30]:
simple_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | simple_prompt
    | groq_llm
    | StrOutputParser()
)

print("Simple LCEL RAG chain created.")

Simple LCEL RAG chain created.


## 33. Run a first simple RAG question

This invokes retrieval and answer generation together. We print the answer only here; the next cell separately inspects retrieved evidence so we can distinguish retrieval problems from generation problems.

In [31]:
first_rag_question = "What is the difference between AI and machine learning?"
first_rag_answer = simple_rag_chain.invoke(first_rag_question)

print("Question:", first_rag_question)
print("\nAnswer:\n", first_rag_answer)

Question: What is the difference between AI and machine learning?

Answer:
 AI (Artificial Intelligence) is the broad field concerned with creating machines that simulate human intelligence—systems that can think, reason, and mimic human actions. Machine learning (ML) is a **subset of AI** that specifically enables those systems to learn from data rather than being explicitly programmed; ML algorithms discover patterns in data to make predictions or decisions. In short, AI encompasses any approach to intelligent behavior, while machine learning focuses on data‑driven learning methods within AI.


## 34. Inspect the context retrieved for that question

This is an important debugging step. If the answer is poor, ask whether the retriever returned relevant chunks first. A fluent answer cannot repair missing or irrelevant evidence by itself.

In [32]:
evidence_docs = retriever.invoke(first_rag_question)
print("Retrieved source documents:", len(evidence_docs))
for rank, doc in enumerate(evidence_docs, start=1):
    print(f"\n--- Evidence {rank} ---")
    print("Source:", doc.metadata.get("source", "Unknown"))
    print("Topic:", doc.metadata.get("topic", "Unknown"))
    print(doc.page_content)

Retrieved source documents: 3

--- Evidence 1 ---
Source: ML Basics
Topic: ML
Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.

--- Evidence 2 ---
Source: AI Introduction
Topic: AI
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.

--- Evidence 3 ---
Source: Deep Learning
Topic: DL
Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.


## 35. Build a streaming RAG chain

The instructor's streaming chain keeps the same retrieval, context formatting, and prompt, but **does not add `StrOutputParser()` at the end**. This allows callers to receive model message chunks as they are generated rather than waiting for one final string.

Streaming changes how output arrives; it does not change the retrieval algorithm or automatically improve answer correctness.

In [33]:
streaming_rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | simple_prompt
    | groq_llm
)

print("Streaming RAG chain created.")

Streaming RAG chain created.


## 36. Stream the generated answer token/chunk by chunk

`stream()` yields model chunks. In common LangChain chat integrations, chunks expose `.content`; we print each chunk immediately with `flush=True` so the output appears progressively in notebook frontends that support streaming.

In [34]:
stream_question = "Explain deep learning in simple terms."
print("Question:", stream_question)
print("\nStreaming answer:\n", end="", flush=True)

for chunk in streaming_rag_chain.stream(stream_question):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()

Question: Explain deep learning in simple terms.

Streaming answer:
Deep learning is a kind of machine learning that works with artificial neural networks—computer models loosely inspired by the brain.  
Instead of being told exactly what to look for, these networks have many layers that automatically pull out simple patterns from raw data (like pixels or sound) and then combine them into higher‑level features. By stacking these layers, deep learning can learn complex representations on its own, which is why it has driven major advances in areas such as computer vision, natural‑language processing, and speech recognition.


## 37. Test the simple and streaming RAG variants together

This helper preserves the instructor's `test_rag_chains(question)` idea. It deliberately calls the model twice: once for a final string answer and once for streamed chunks. That means this demonstration can consume two generation requests per question.

In [35]:
def test_rag_chains(question: str) -> None:
    """Run both final-string and streaming variants for one question."""
    print("Question:", question)
    print("=" * 80)

    print("\n1. Simple RAG chain:")
    answer = simple_rag_chain.invoke(question)
    print(answer)

    print("\n2. Streaming RAG chain:")
    for chunk in streaming_rag_chain.stream(question):
        if chunk.content:
            print(chunk.content, end="", flush=True)
    print("\n")

## 38. Run the chain-test helper once

The instructor demonstrates both simple and streaming output with one question. Start with one request before testing many questions, especially if your API plan has rate limits or cost controls.

In [36]:
test_rag_chains("How does NLP work?")

Question: How does NLP work?

1. Simple RAG chain:
Natural Language Processing works by bringing together **computational linguistics**—the rules and structures of human language—with **machine‑learning and deep‑learning models**. These models are trained on large amounts of text data, allowing the system to learn patterns, meanings, and relationships in language without being explicitly programmed for every rule. By extracting and interpreting these patterns, NLP enables computers to understand and manipulate human language for tasks such as chatbots, translation, sentiment analysis, and text summarization.

2. Streaming RAG chain:
Natural Language Processing works by bringing together **computational linguistics**—the rules and structures of human language—with **machine‑learning and deep‑learning models**. These models are trained on large amounts of text data, allowing the system to recognize patterns, extract meaning, and perform tasks such as translation, sentiment analysis, chat

## 39. Test several questions through simple RAG

The instructor uses three test questions to cover separate concepts. This is a **smoke test**, not a formal evaluation benchmark: it does not measure recall, answer correctness, faithfulness, or latency on a representative dataset.

In [37]:
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms.",
    "How does NLP work?",
]

for question in test_questions:
    print("\n" + "=" * 80)
    print("Question:", question)
    print(simple_rag_chain.invoke(question))


Question: What is the difference between AI and Machine Learning?
AI (Artificial Intelligence) is the broad field that aims to simulate human intelligence in machines, enabling them to think, reason, and act like humans. It encompasses any approach that tries to mimic human cognitive abilities, whether through rule‑based systems, expert systems, or learning methods, and is divided into narrow AI and general AI.

Machine Learning (ML) is a specific subset of AI. It focuses on giving computers the ability to learn from data rather than being explicitly programmed. ML algorithms discover patterns in data and improve their performance on tasks through experience, covering methods such as supervised, unsupervised, and reinforcement learning.

**Key difference:** AI is the overall discipline of creating intelligent behavior, while Machine Learning is a particular technique within AI that achieves intelligence by learning from data.

Question: Explain deep learning in simple terms.
Deep lear

## 40. Conversational RAG: prompt with chat history

A conversational prompt contains a `MessagesPlaceholder` where previous `HumanMessage` and `AIMessage` objects can be inserted. The prompt receives:

- `chat_history`: messages from earlier turns.
- `context`: documents retrieved for the current user input.
- `input`: the current user question.

**Important limitation:** the instructor's simple chain sends `input` directly to the retriever and places history in the answer prompt. It does not rewrite ambiguous follow-up text before retrieval. For “What are its main types?” it may retrieve poorly because the retriever sees that short question without resolving “its.” A history-aware query-rewrite retriever is a possible next step, but it is not silently substituted into this lesson.

In [38]:
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer using the provided context. If context is insufficient, say so."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {input}"),
])

## 41. Build the conversational RAG chain

`RunnablePassthrough.assign(...)` keeps the input dictionary and adds `context` computed from the current `input`. Then the conversational prompt combines context, current question, and prior messages; Groq generates the answer; `StrOutputParser()` returns plain text.

This chain does not automatically store history for later calls. The caller owns the `chat_history` list and must append each exchange.

In [39]:
conversational_rag_chain = (
    RunnablePassthrough.assign(
        context=lambda inputs: format_docs(retriever.invoke(inputs["input"]))
    )
    | conversational_prompt
    | groq_llm
    | StrOutputParser()
)

print("Conversational RAG chain created.")

Conversational RAG chain created.


## 42. Start a conversation with empty history

The first question establishes the conversation. We pass `chat_history=[]` explicitly because the prompt requires that variable. The chain returns a text answer, rather than a dictionary, because `StrOutputParser()` is the final component.

In [40]:
chat_history = []
q1 = "What is machine learning?"
a1 = conversational_rag_chain.invoke({
    "input": q1,
    "chat_history": chat_history,
})

print("Q1:", q1)
print("A1:", a1)

Q1: What is machine learning?
A1: Machine learning (ML) is a branch of artificial intelligence that enables systems to learn from data rather than being explicitly programmed. ML algorithms discover patterns in data to make predictions or decisions, and they are typically categorized into supervised, unsupervised, and reinforcement learning approaches. (Source: *ML Basics*)


## 43. Add the first turn to chat history

The human message represents the user's question; the AI message represents the assistant's answer. Passing these typed message objects lets the placeholder preserve conversational roles. This list exists in the current Python kernel only and is not durable application memory.

In [41]:
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1),
])

print("Messages stored in history:", len(chat_history))
for message in chat_history:
    print(type(message).__name__ + ":", message.content[:180])

Messages stored in history: 2
HumanMessage: What is machine learning?
AIMessage: Machine learning (ML) is a branch of artificial intelligence that enables systems to learn from data rather than being explicitly programmed. ML algorithms discover patterns in dat


## 44. Ask a follow-up question

The instructor's follow-up is “How is it different from traditional programming?” We pass the same history along with the new input. The answer prompt now sees the previous exchange, but the raw follow-up is still the string sent to the retriever. This is why conversational context helps generation, while query rewriting may still be needed for reliable follow-up retrieval.

In [42]:
q2 = "How is it different from traditional programming?"
a2 = conversational_rag_chain.invoke({
    "input": q2,
    "chat_history": chat_history,
})

print("Q2:", q2)
print("A2:", a2)
print("\nNote: chat history was included in the prompt; retrieval still used the raw follow-up question.")

Q2: How is it different from traditional programming?
A2: Machine learning differs from traditional (explicit) programming in that **the logic isn’t hand‑written by a developer**. In a conventional program you specify step‑by‑step instructions that tell the computer exactly how to handle every possible input.  

In machine‑learning systems, you instead **feed the algorithm lots of data** and let it **discover patterns and build a model** that can make predictions or decisions on new, unseen data. As Document 1 (ML Basics) states, “Instead of being explicitly programmed, ML algorithms find patterns in data.” This data‑driven approach lets the system improve its behavior automatically as more data become available, whereas traditional code would require the programmer to manually update the logic.

Note: chat history was included in the prompt; retrieval still used the raw follow-up question.


## 45. Interview-ready vocabulary and definitions

| Term | Clear definition |
|---|---|
| RAG | A system that retrieves external information and supplies it as context to a generative model at answer time. |
| Embedding | A numeric vector produced by a model to represent content in a space useful for similarity/retrieval tasks. |
| Dense vector | A numerical representation in which most coordinates can be non-zero. |
| `embed_documents()` | The embedding interface method used to create a vector for each indexed source text. |
| `embed_query()` | The embedding interface method used to create a vector for a search query. |
| Chunk | A smaller passage split from a source document and used as a retrieval unit. |
| Metadata | Structured attributes attached to a document, such as its source, page, or topic. |
| FAISS | A library for indexing and searching vectors; not by itself a full managed vector-database service. |
| Index | A data structure designed to support vector search under a particular metric/algorithm. |
| Nearest-neighbor search | Finding indexed vectors closest to a query vector under a specified metric. |
| L2 distance | Euclidean distance between vectors; lower distance means closer vectors. The default FAISS flat L2 path reports a squared L2 distance. |
| Dot product | Sum of coordinate-wise products of two vectors; with normalized vectors it equals cosine similarity. |
| Cosine similarity | Dot product divided by the product of the vectors' magnitudes, for non-zero vectors. |
| Top-k | Return up to the `k` highest-ranked candidate documents from search. |
| Distance score | A metric-specific distance; it is not automatically a probability, percentage, or factuality score. |
| Metadata filter | A structured predicate that limits eligible records during retrieval. |
| Retriever | A standard interface that accepts a query and returns relevant documents. |
| Vector store | A component that stores vectors and associated records and exposes vector-search methods. |
| LCEL | LangChain Expression Language for composing runnable components into chains. |
| `RunnablePassthrough` | A runnable that passes its input forward; in this chain it carries the original question into the prompt. |
| `format_docs()` | A helper that turns a list of retrieved `Document` objects into a prompt-ready context string. |
| `StrOutputParser` | Converts a model message output into plain text. |
| Streaming | Delivering generated output incrementally as chunks rather than waiting for a complete answer. |
| Conversational RAG | RAG where the prompt can also receive prior conversation messages. It does not automatically imply query rewriting or persistent memory. |
| Pickle deserialization | Loading Python objects from a pickle artifact; loading untrusted pickle data can execute code. |

**30-second system explanation:** “I load the source texts into LangChain `Document` objects, preserve source/topic metadata, and split text into passages. I use Gemini Embedding 2 to generate document vectors and build a local FAISS index. At query time, the same embedding adapter vectorizes the question and FAISS retrieves the top passages. I format those passages into the prompt and use a Groq-hosted chat model to generate the answer. LCEL composes the steps; alternate chains demonstrate streaming and a prompt that includes chat history.”

## 46. Troubleshooting checklist

1. **`GEMINI_API_KEY` missing:** verify `.env` location and spelling; do not print the key.
2. **Gemini model/API error:** test the direct embedding cell first; verify model ID, access, quota, and network.
3. **Dimension mismatch:** never mix vectors from different models/dimensions in one index. Re-embed and rebuild the index.
4. **FAISS import/install error:** verify the Python version and platform wheel; restart the notebook kernel after dependency changes.
5. **No results or weak results:** inspect `chunks`, query text, source metadata, and returned passages. Change chunking only after reviewing failures.
6. **Scores look “high” or “low”:** identify whether the value is distance or similarity. In this default L2 setup, lower distance is closer.
7. **Metadata filter returns too few documents:** confirm the metadata key/value and raise `fetch_k` if filtering occurs after candidate retrieval.
8. **`load_local()` errors:** confirm the path exists and the same compatible embedding adapter is supplied. Enable dangerous deserialization only for your trusted locally created index.
9. **Groq API/model error:** validate `GROQ_API_KEY`, the configured model ID, provider quota, and current model availability in the separate Groq cell.
10. **Bad RAG answer:** inspect retrieval evidence before editing the generation prompt; the model cannot reliably answer from evidence it did not receive.
11. **Conversation follow-up confusion:** inspect `chat_history`; remember this implementation sends the raw follow-up to retrieval and only adds history to the generation prompt.
12. **Old conversation disappears:** this notebook's Python list is in-memory only; durable session memory needs an external store or a stateful application framework.

## 47. Practical exercises — prove that you understand the lesson

Complete these without copying a finished answer:

1. Add a fifth `Document` with a unique `topic`, rebuild the chunks and index, and prove that metadata filtering finds it.
2. Change `chunk_size` from 500 to 200. Compare chunk count, source metadata, and the actual text returned for the same query.
3. Explain why the query prefix differs from the document prefix for Gemini Embedding 2. What must be rebuilt if you change the embedding model?
4. Compare cosine similarity from `compare_embeddings()` with the default FAISS L2 distance. Explain why their numbers should not be compared as if they were the same scale.
5. Save, restart the kernel, recreate the Gemini adapter and load the FAISS index. Explain why loading requires an embedding interface even though the vectors are already on disk.
6. Increase `k` from 3 to 4 and inspect whether the additional chunk adds evidence or distracts the answer.
7. Add a question that is not answerable from the corpus. Does the model admit the evidence is insufficient? Test multiple formulations.
8. Change the simple RAG chain to return both the answer and the retrieved documents. Explain how that helps debugging.
9. Compare final-string and streaming output. What exactly changes, and what stays the same?
10. Improve conversational retrieval by rewriting a follow-up question into a standalone query before invoking the retriever. Label that as an extension beyond the instructor's baseline chain.

### Done means

You can describe every stage out loud, code it from memory, inspect failed retrievals, explain scores accurately, and identify the security risks of loading index files. A successful smoke test is not proof of production quality.

## Official references and source notes

- **Instructor source notebook:** `2-faiss.ipynb` (46 cells, including the final blank cell).
- **Transcript 1:** `Pasted text(20261026-085833).txt` — FAISS introduction, data ingestion, chunking, embeddings, cosine similarity, building/saving/loading the vector store, search, scores, filters, and transition to RAG chains.
- **Transcript 2:** `Pasted text (2)(20261026-085932).txt` — Groq model setup, prompt construction, retriever, `format_docs`, LCEL basic RAG, conversational RAG, streaming RAG, single/multiple query tests, chat history update, and follow-up question.
- FAISS project and description: https://github.com/facebookresearch/faiss
- FAISS LangChain integration: https://docs.langchain.com/oss/python/integrations/vectorstores/faiss
- FAISS CPU package: https://pypi.org/project/faiss-cpu/
- Google Gemini embeddings: https://ai.google.dev/gemini-api/docs/embeddings
- Groq supported models: https://console.groq.com/docs/models
- Groq GPT-OSS 120B model page: https://console.groq.com/docs/model/openai/gpt-oss-120b
- LangChain Groq integration package: https://python.langchain.com/docs/integrations/chat/groq/

### Source coverage crosswalk

| Original instructor cell(s) | Subject | Where retained in this notebook |
|---|---|---|
| 0–1 | Intro to RAG; FAISS definition, speed/memory, CPU/GPU, dense-vector search | Opening cells and Sections 5, 45–47 |
| 2 | Imports and environment loading | Sections 1–4; shared imports and provider-specific sections |
| 3–6 | Data ingestion; `Document` page content/metadata; recursive splitting; inspect chunks | Sections 6–10 |
| 7–10 | OpenAI key/model; query embedding; batch document embeddings | Replaced by Gemini key/client/direct embedding, adapter, and batch examples in Sections 11–16 |
| 11–14 | Cosine similarity; AI vs artificial intelligence; AI vs pizza; ML vs ML | Sections 17–18 |
| 15–17 | Create the FAISS vector store and inspect it | Sections 19–20 |
| 18–19 | Save and load local vector store | Sections 21–23, with pickle-safety warning |
| 20–21 | Top-k similarity search and print sources/content | Section 24 |
| 22–23 | Similarity search with score; inspect chunks | Sections 25 and 10; clarifies distance semantics |
| 24–25 | Filter by `topic="ML"` | Section 26 |
| 26–28 in the source notebook | Initialize Groq/chat model and smoke-test it | Sections 28–29, using a current model ID and separate Groq API setup |
| 29–34 | Simple RAG prompt, retriever, formatting, LCEL chain | Sections 30–34 |
| 35–37 | Conversational prompt with chat history and chain | Sections 40–44 |
| 38–41 | Streaming chain, helper test, one and multiple question examples | Sections 35–39 |
| 42–44 | First conversation, update history, follow-up question | Sections 42–44 |
| 45 | Empty cell | Removed |

### Source-derived vs improved

- The core data and topic examples, function names, search/filter operations, and three chain patterns follow the instructor's notebook/transcripts.
- OpenAI-specific executable calls were replaced by Gemini embedding calls because that is the requested provider setup.
- Groq usage is retained but isolated from the Gemini code and moved to a dedicated key/model/smoke-test section. The model ID is a Groq-supported model identifier, and the request uses the Groq API.
- The source's `separators=[" "]` was replaced with recursive paragraph/line/space/character separators so splitting behaves as expected.
- Score values are described as distance under the default L2 path, not a universal similarity percentage.
- The saved local artifact's pickle is called out as unsafe when untrusted.
- Conversational history is accurately described as a prompt input, not as persistent memory and not as automatic query rewriting.

### Runtime status

This is a teaching artifact. Syntax/structure checks are performed outside the notebook, but the complete notebook is **not pre-executed** because Gemini and Groq calls require your credentials and may consume quota. Run all cells from top to bottom in your configured environment, then address any live provider/model/version errors with the separate smoke-test cells.